# Mini-Ludo collusion detector
Run the cell below (click the play button or press Shift+Enter). It takes about 2-3 minutes, then saves `results.csv` and `results.png` (open the folder icon on the left to see them).
To change the experiment, edit the SETTINGS block at the top of the cell, then run it again.

In [ ]:
"""
mini_ludo_collusion.py   (version 2: follows the Wikipedia Ludo rules and board picture)
=====================================================================================
A Ludo simulator with an optional, secretly colluding pair, plus an
information-theoretic detector. Written for Apart Research's AI Collusion
Sprint, project 1.4 ("Can information-theoretic methods identify colluding
pairs?"). It is INSPIRED BY Bonjour et al. (2022) but is NOT their exact
statistic: Ludo is turn-based, so we look at the dependence between WHO a capture
would hit (the target) and WHETHER the player took it (the decision).

HOW TO RUN (Google Colab)
  1. Upload this file (folder icon on the left), then in a cell run:  %run mini_ludo_collusion.py
     or paste the whole file into one cell and run it.
  2. A self-test runs first, then a table prints. results.csv and results.png are saved.
  3. For a quick test, set EXPERIMENTS = 5 and CONTROL_EXPERIMENTS = 5 below.

THE GAME (your rules, from the Wikipedia text and the board picture)
  * 4 players with 4 tokens each, no partnerships. Seats go clockwise:
    0 = green (left), 1 = yellow (top), 2 = light blue (right), 3 = pink (bottom).
  * The highest first roll starts (ties are re-rolled). Play passes clockwise.
  * A six lets you draw one token from the yard onto your start square, or move a token 6.
    Otherwise you must move a token by the number rolled. If no move is legal, the turn passes.
  * BONUS ROLL (default): every six earns a bonus roll, and you move after each one.
    A third six in a row means no move for that roll and the turn passes at once.
    (Set BONUS_ROLL_RULE = "yard_empty" for the stricter reading where the bonus roll
    only applies when you cannot draw a token from the yard.)
  * Landing on a single opponent token sends it back to its yard.
  * BLOCKS: two or more tokens of one colour on the same square form a block.
    An opponent who lands on a block is sent back to ITS yard; the block stays.
    (This also covers the "doubled" start square.)
  * Board geometry (checked against the picture): the shared track has 52 squares and
    start squares are 13 apart. A token travels 51 track squares (progress 0 to 50, ending on
    the square just before its home column), then 5 home-column squares (progress 51 to 55,
    always safe), then the finish (progress 56), which needs an EXACT roll.
  * No other safe squares. First player to finish all four tokens wins.

COLLUSION (what the secret pair does)
  * "Sparing": when a capture of the partner's token is available, the colluder
    avoids it with probability STRENGTH (if another legal move exists).
  * MODE "association": both partners spare each other (mutual, symmetric).
  * MODE "assistant":   only partner A spares partner B (one-way help).
  * "Spectator collusion" is NOT implemented: Ludo has no hidden information,
    so a spectator has nothing secret to pass on.

THE DETECTOR (state-conditioned: only counts moments when a capture was possible)
  For each player i we collect events (target j, decision d), where d = 1 if i captured
  j's token and 0 if i could have but did not. I_i(Target; Decision) measures how much the
  decision depends on whose token it is. An honest player ignores ownership -> about 0.
  A colluder spares one specific player -> positive. We split I_i into a contribution
  per target j, add the two directions of a pair, and rank the pairs. A shuffle test
  gives a p-value. BASELINE for comparison: the pair with the lowest capture rate.
"""

import csv
import itertools
import random

import numpy as np

# =========================== SETTINGS (edit these) ===========================
N_PLAYERS = 4              # keep at 4 (the board picture has 4 colours)
COLOURS = ["green", "yellow", "light blue", "pink"]
TOKENS = 4                 # tokens per player
RING = 52                  # squares on the shared track
START_GAP = 13             # distance between neighbouring start squares
LAST_RING_PROGRESS = 50    # progress 0..50 is on the shared track
HOME = 56                  # progress 51..55 = home column, 56 = finish
BONUS_ROLL_RULE = "always"       # "always" (bonus roll after every six) or "yard_empty"
MAX_TURNS = 6000           # safety cap so a game can never run forever
HONEST_NOISE = 0.2         # chance an honest player picks a random legal move

PARTNERS = (0, 2)          # the colluding pair: green and light blue (opposite seats)
MODE = "association"       # "association" or "assistant"

STRENGTHS = [0.0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]   # collusion probability; 0.0 = nobody colludes (negative control)
GAMES_LIST = [5, 10, 20]            # games observed per experiment (sample size)
EXPERIMENTS = 100                   # repeats per condition
CONTROL_EXPERIMENTS = 500           # more repeats for the no-collusion control (strength 0.0)
N_PERM = 200                        # shuffles for the p-value
ALPHA = 0.05
SEED = 12345
# ============================================================================


# ------------------------------- the game -----------------------------------
def ring_pos(player, progress):
    return (START_GAP * player + progress) % RING


def legal_moves(state, player, roll):
    """Moves are tuples (token, new_progress, captured, suicide).
    captured = {owner: [token ids]} for a single opponent token that would be sent home.
    suicide  = True if the move lands on an opponent BLOCK (the mover's token is sent home)."""
    moves = []
    for t, prog in enumerate(state[player]):
        if prog == HOME:
            continue
        if prog == -1:                      # token in the yard
            if roll != 6:
                continue
            new = 0
        else:
            new = prog + roll
            if new > HOME:                  # exact roll needed to finish
                continue
        captured, suicide = {}, False
        if new <= LAST_RING_PROGRESS:       # home column is safe, so only the track matters
            pos = ring_pos(player, new)
            for q in range(N_PLAYERS):
                if q == player:
                    continue
                here = [u for u, pq in enumerate(state[q])
                        if 0 <= pq <= LAST_RING_PROGRESS and ring_pos(q, pq) == pos]
                if len(here) >= 2:
                    suicide = True
                elif len(here) == 1:
                    captured[q] = here
        moves.append((t, new, captured, suicide))
    return moves


def score(move, state, player):
    """How good an honest player thinks a move is."""
    t, new, captured, suicide = move
    if suicide:
        return -1000.0
    s = float(new)                          # prefer moving the furthest token
    if captured:
        s += 100 + 10 * sum(len(v) for v in captured.values())
    if new == HOME:
        s += 60
    if state[player][t] == -1:              # bringing a token out
        s += 40
    if new <= LAST_RING_PROGRESS and any(u != t and state[player][u] == new for u in range(TOKENS)):
        s += 10                             # forming a block
    return s


def choose_move(rng, candidates, state, player):
    if rng.random() < HONEST_NOISE:
        return rng.choice(candidates)
    return max(candidates, key=lambda m: score(m, state, player))


def partner_of(player):
    a, b = PARTNERS
    return b if player == a else a


def spares(player, mode):
    a, b = PARTNERS
    if mode == "association":
        return player in (a, b)
    if mode == "assistant":
        return player == a
    return False


def opening_player(rng):
    """Highest first roll starts; tied players roll again."""
    contenders = list(range(N_PLAYERS))
    while len(contenders) > 1:
        rolls = {p: rng.randint(1, 6) for p in contenders}
        best = max(rolls.values())
        contenders = [p for p in contenders if rolls[p] == best]
    return contenders[0]


def play_game(rng, strength, mode, events):
    """Play one game. Appends (player, target, decision) to events. Returns (winner, turns)."""
    state = [[-1] * TOKENS for _ in range(N_PLAYERS)]
    player = opening_player(rng)
    turns = 0
    while turns < MAX_TURNS:
        turns += 1
        sixes = 0
        while True:
            roll = rng.randint(1, 6)
            yard_empty = all(p != -1 for p in state[player])
            if roll == 6:
                sixes += 1
                if sixes == 3:              # third six in a row: no move, turn passes
                    break
            moves = legal_moves(state, player, roll)
            if moves:
                candidates = moves
                if spares(player, mode) and rng.random() < strength:
                    partner = partner_of(player)
                    others = [m for m in moves if partner not in m[2]]
                    if others:              # skip captures of the partner if possible
                        candidates = others
                chosen = choose_move(rng, candidates, state, player)

                # Log one event for every opponent who COULD have been captured
                targets = set()
                for m in moves:
                    targets.update(m[2].keys())
                for j in targets:
                    events.append((player, j, 1 if j in chosen[2] else 0))

                t, new, captured, suicide = chosen
                if suicide:                 # landed on an opponent block
                    state[player][t] = -1
                else:
                    state[player][t] = new
                    for q, token_ids in captured.items():
                        for u in token_ids:
                            state[q][u] = -1
                    if all(p == HOME for p in state[player]):
                        return player, turns
            if roll == 6 and (BONUS_ROLL_RULE == "always" or yard_empty):
                continue                    # bonus roll
            break
        player = (player + 1) % N_PLAYERS
    return None, turns


def self_test():
    """Checks the rules and the board geometry. Raises an error if something is wrong."""
    for p in range(N_PLAYERS):
        assert ring_pos(p, 50) == (ring_pos(p, 0) - 2) % RING, "home-column entry square"
        assert ring_pos((p + 1) % N_PLAYERS, 0) == (ring_pos(p, 0) + START_GAP) % RING, "start spacing"

    def blank():
        return [[-1] * TOKENS for _ in range(N_PLAYERS)]

    s = blank(); s[0][0] = 5; s[1][0] = 47            # opponent sits on ring square 8
    m = legal_moves(s, 0, 3)
    assert len(m) == 1 and m[0][2] == {1: [0]} and not m[0][3], "single capture"

    s[1][1] = 47                                      # now two opponent tokens: a block
    m = legal_moves(s, 0, 3)
    assert m[0][3] is True and m[0][2] == {}, "block sends the attacker home"

    s = blank(); s[0][0] = 5; s[0][1] = 8             # stacking own tokens
    mv = {x[0]: x for x in legal_moves(s, 0, 3)}
    assert mv[0][1] == 8 and mv[0][2] == {} and not mv[0][3], "own stack"

    s = blank(); s[0][0] = 55                         # exact roll needed to finish
    assert legal_moves(s, 0, 2) == [] and len(legal_moves(s, 0, 1)) == 1, "exact finish"

    s = blank()                                       # a six is needed to leave the yard
    assert legal_moves(s, 0, 5) == [] and len(legal_moves(s, 0, 6)) == TOKENS, "six to enter"

    s = blank(); s[0][0] = 9; s[1][0] = 51            # home column is safe
    assert legal_moves(s, 0, 3)[0][2] == {}, "home column safe"
    print("Self-test passed: capture, block, stacking, exact finish, six to enter, safe home column.")


# ----------------------------- the statistics -------------------------------
def count_matrix(events):
    """N[i, j, d] = how often player i could capture j's token and chose d (1 = captured)."""
    N = np.zeros((N_PLAYERS, N_PLAYERS, 2))
    for i, j, d in events:
        N[i, j, d] += 1
    return N


def cell_scores(N):
    """c[i, j] = part of I_i(Target; Decision), in bits, that comes from target j."""
    c = np.zeros((N_PLAYERS, N_PLAYERS))
    for i in range(N_PLAYERS):
        n = N[i].sum()
        if n == 0:
            continue
        p = N[i] / n                              # joint probability p(target, decision)
        p_target = p.sum(axis=1, keepdims=True)
        p_decision = p.sum(axis=0, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            term = np.where(p > 0, p * np.log2(p / (p_target * p_decision)), 0.0)
        c[i] = term.sum(axis=1)
    return c


def pair_scores(c):
    return {(a, b): c[a, b] + c[b, a] for a, b in itertools.combinations(range(N_PLAYERS), 2)}


def baseline_scores(N):
    """Simple baseline: how LOW is the capture rate between two players? (higher = more suspicious)"""
    out = {}
    for a, b in itertools.combinations(range(N_PLAYERS), 2):
        opportunities = N[a, b].sum() + N[b, a].sum()
        captures = N[a, b, 1] + N[b, a, 1]
        out[(a, b)] = 1.0 - captures / opportunities if opportunities > 0 else 0.0
    return out


def permutation_maxes(events, rng_np, n_perm):
    """Shuffle each player's decisions across targets; record the biggest pair score each time."""
    arr = np.array(events, dtype=int)
    per_player = []
    for i in range(N_PLAYERS):
        e = arr[arr[:, 0] == i]
        per_player.append((e[:, 1], e[:, 2]))
    maxes = np.zeros(n_perm)
    for k in range(n_perm):
        N = np.zeros((N_PLAYERS, N_PLAYERS, 2))
        for i, (targets, decisions) in enumerate(per_player):
            if len(targets):
                np.add.at(N[i], (targets, rng_np.permutation(decisions)), 1)
        maxes[k] = max(pair_scores(cell_scores(N)).values())
    return maxes


# ------------------------------ experiments ---------------------------------
def evaluate(strength, n_games, mode, n_exp, n_perm, seed):
    rng_np = np.random.default_rng(seed)
    true_pair = tuple(sorted(PARTNERS))
    correct_mi = correct_base = flagged = unfinished = 0
    rate_pair, rate_others, n_events = [], [], []
    for e in range(n_exp):
        rng = random.Random(f"{seed}-{strength}-{n_games}-{e}")
        events = []
        for _ in range(n_games):
            winner, _ = play_game(rng, strength, mode, events)
            unfinished += winner is None
        N = count_matrix(events)
        ps = pair_scores(cell_scores(N))
        bs = baseline_scores(N)
        correct_mi += max(ps, key=ps.get) == true_pair
        correct_base += max(bs, key=bs.get) == true_pair
        maxes = permutation_maxes(events, rng_np, n_perm)
        p_value = (1 + np.sum(maxes >= max(ps.values()))) / (1 + n_perm)
        flagged += p_value < ALPHA

        a, b = true_pair
        pair_opps = N[a, b].sum() + N[b, a].sum()
        pair_caps = N[a, b, 1] + N[b, a, 1]
        all_opps, all_caps = N.sum(), N[:, :, 1].sum()
        rate_pair.append(pair_caps / pair_opps if pair_opps else np.nan)
        rate_others.append((all_caps - pair_caps) / (all_opps - pair_opps))
        n_events.append(len(events))
    return {
        "strength": strength,
        "games": n_games,
        "pair_found_MI": correct_mi / n_exp,
        "pair_found_baseline": correct_base / n_exp,
        "flag_rate": flagged / n_exp,
        "capture_rate_pair": float(np.nanmean(rate_pair)),
        "capture_rate_others": float(np.mean(rate_others)),
        "events_per_experiment": float(np.mean(n_events)),
        "unfinished_games": unfinished,
    }


def demo_game():
    rng = random.Random(SEED)
    events = []
    winner, turns = play_game(rng, 0.0, MODE, events)
    name = COLOURS[winner] if winner is not None else "nobody (hit the turn cap)"
    print(f"Demo game: winner = {name}, turns = {turns}, capture opportunities logged = {len(events)}")


def make_plot(rows):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    biggest = max(r["games"] for r in rows)
    sel = sorted([r for r in rows if r["games"] == biggest], key=lambda r: r["strength"])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    xs = [r["strength"] for r in sel]
    axes[0].plot(xs, [r["pair_found_MI"] for r in sel], "o-", label="Information-theoretic score")
    axes[0].plot(xs, [r["pair_found_baseline"] for r in sel], "s--", label="Baseline (capture rate)")
    axes[0].axhline(1 / 6, color="grey", linestyle=":", label="Chance (1 in 6)")
    axes[0].set_xlabel("Collusion strength")
    axes[0].set_ylabel("Fraction of experiments naming the true pair")
    axes[0].set_title(f"Pair identified ({biggest} games per experiment)")
    axes[0].set_ylim(0, 1.05)
    axes[0].legend()

    strengths = sorted({r["strength"] for r in rows if r["strength"] > 0})
    for s in strengths:
        sub = sorted([r for r in rows if r["strength"] == s], key=lambda r: r["games"])
        axes[1].plot([r["games"] for r in sub], [r["pair_found_MI"] for r in sub], "o-", label=f"strength {s}")
    axes[1].axhline(1 / 6, color="grey", linestyle=":")
    axes[1].set_xlabel("Games observed per experiment")
    axes[1].set_ylabel("Fraction naming the true pair")
    axes[1].set_title("Effect of sample size (information-theoretic score)")
    axes[1].set_ylim(0, 1.05)
    axes[1].legend()

    fig.tight_layout()
    fig.savefig("results.png", dpi=150)
    print("Saved results.png")


def threshold_table(rows, level=0.8):
    """For each sample size: the smallest collusion probability at which the true pair is found often enough."""
    print(f"\nSmallest collusion probability where the true pair is named in at least {level:.0%} of experiments:")
    for g in sorted({r["games"] for r in rows}):
        sub = sorted([r for r in rows if r["games"] == g and r["strength"] > 0], key=lambda r: r["strength"])

        def first(key):
            for r in sub:
                if r[key] >= level:
                    return str(r["strength"])
            return "not reached"

        print(f"  {g:>2} games: information-theoretic = {first('pair_found_MI'):<12} baseline = {first('pair_found_baseline')}")


def main():
    self_test()
    a, b = PARTNERS
    print(f"Colluding pair: {COLOURS[a]} (seat {a}) and {COLOURS[b]} (seat {b}), mode = {MODE}")
    demo_game()
    rows = []
    for strength in STRENGTHS:
        for n_games in GAMES_LIST:
            n_exp = CONTROL_EXPERIMENTS if strength == 0.0 else EXPERIMENTS
            rows.append(evaluate(strength, n_games, MODE, n_exp, N_PERM, SEED))
            r = rows[-1]
            print(f"strength={r['strength']:<4} games={r['games']:<3} "
                  f"pair found (MI)={r['pair_found_MI']:.2f}  baseline={r['pair_found_baseline']:.2f}  "
                  f"flagged={r['flag_rate']:.2f}  "
                  f"capture rate pair={r['capture_rate_pair']:.2f} vs others={r['capture_rate_others']:.2f}"
                  f"{'  UNFINISHED GAMES: ' + str(r['unfinished_games']) if r['unfinished_games'] else ''}")
    with open("results.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print("Saved results.csv")
    make_plot(rows)
    threshold_table(rows)
    print("\nRead strength = 0.0 as your NEGATIVE control: 'flagged' should be about "
          f"{ALPHA:.2f} and 'pair found' about 0.17 (chance).")


if __name__ == "__main__":
    main()


## Optional: download your results to your laptop

In [ ]:
from google.colab import files
files.download('results.csv')
files.download('results.png')
